# Notebook 3: Prompt Management con Langfuse

## Objetivos
- Entender por qué el prompt es un **artefacto versionable** (como el código fuente con Git)
- Comprender el **problema del prompt drift** y sus consecuencias en producción
- Aplicar las **5 secciones de un buen system prompt**: Persona, Alcance, Herramientas, Formato, Anti-alucinación
- Crear y versionar prompts en Langfuse con labels (`production`, `staging`, `latest`)
- Usar el patrón `fallback` para garantizar disponibilidad cuando Langfuse no está accesible
- Usar `{{variables}}` para hacer el prompt parametrizable en tiempo de ejecución
- Vincular prompts a trazas para ver métricas **por versión de prompt** en Langfuse
- Dominar el flujo completo: crear → evaluar → promover → rollback

## El problema: Prompt Drift

En el Notebook 1 construimos el agente con un system prompt **hardcodeado** en `config.py`. Funciona, pero tiene problemas graves para cualquier equipo o entorno de producción:

| Problema | Consecuencia |
|----------|-------------|
| **Sin historial de cambios** | No sabes qué versión del prompt estaba activa cuando surgió un bug |
| **Deploy completo para un cambio de texto** | Cambiar una frase requiere commit → PR → merge → deploy completo |
| **Sin rollback rápido** | Si el nuevo prompt empeora el agente, revertir requiere otro deploy (minutos u horas) |
| **Sin comparación A/B** | No puedes probar dos versiones en paralelo |
| **Sin métricas por versión** | No sabes si la v2 es mejor o peor que la v1 |
| **Cambios no auditados** | Cualquiera puede editar el string sin revisión ni aprobación |

> **Prompt Drift:** Cuando el comportamiento del agente cambia gradualmente por modificaciones no controladas del prompt — sin registro, sin revisión, sin posibilidad de revertir. Es el equivalente a editar código sin Git.

### Ejemplo real de Prompt Drift en un equipo

```bash
Lunes:     Prompt dice "Responde en español, máximo 3 frases, tono profesional"
           → Agente funciona correctamente. Respuestas cortas y precisas.
Martes:    Dev A añade "Sé amigable y detallado con el cliente"
           → Intent buena. Pero "detallado" contradice "máximo 3 frases".
Miércoles: El agente empieza a responder con párrafos largos.
Jueves:    Dev B añade "Incluye emojis para hacer la experiencia más cercana 😊"
Viernes:   Los usuarios se quejan. El equipo revisa el código.
           → "¿Quién cambió el prompt?" → "No sé, fue un cambio pequeño."
           → "¿Cuándo dejó de funcionar bien?" → "No tenemos datos."
           → "¿Cómo era el prompt que sí funcionaba?" → "No lo guardamos."
```

## La solución: Gestión de Prompts como Código

Un prompt manager resuelve los mismos problemas que Git resuelve para el código:

| Necesidad | Git (para código) | Prompt Manager (para prompts) |
|-----------|-------------------|-------------------------------|
| **Historial** | Historial de commits | Versiones inmutables (v1, v2, v3...) |
| **Deploy** | CI/CD pipeline → merge → deploy | Mover label "production" → efecto inmediato |
| **Rollback** | `git revert` + re-deploy | Mover label a versión anterior (segundos) |
| **Comparación** | `git diff` | Comparar v1 vs v2 en dashboard + métricas |
| **Auditoría** | Historial de quién hizo qué | Historial de quién creó qué versión |

```bash
Desarrollador edita prompt
        │
        ▼
create_prompt(name, content, labels=["staging"])  ← nueva versión en Langfuse
        │
        ▼
Eval suite contra label "staging"
        │
        ├── Pasa → move label "production" a nueva versión  ← deploy en segundos
        └── Falla → label "production" queda en versión anterior  ← rollback automático
```

Referencia: https://langfuse.com/docs/prompt-management/overview

## 1. Setup

> **Prerrequisitos:**: Cuenta en https://cloud.langfuse.com
> 
> **Variables de entorno necesarias** en `.env`:
> ```
> LANGFUSE_PUBLIC_KEY=pk-lf-...
> LANGFUSE_SECRET_KEY=sk-lf-...
> LANGFUSE_BASE_URL=http://localhost:3000   # local Docker
> # LANGFUSE_BASE_URL=https://cloud.langfuse.com  # cloud EU
> ```
> 
> **Nota v4:** La variable se llama `LANGFUSE_BASE_URL`, no `LANGFUSE_HOST` (renombrada).


In [1]:
# Verificar dependencias (instaladas via: cd notebooks/ && uv sync)
import strands, langfuse, boto3

In [ ]:
import json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Carga variables de entorno desde .env (busca subiendo desde el directorio actual)
load_dotenv(find_dotenv(usecwd=True))

# SDK v4: usar get_client() en vez de Langfuse()
# get_client() devuelve un singleton — la misma instancia en toda la sesión.
# Lee LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY y LANGFUSE_BASE_URL del entorno.
# Si las credenciales están ausentes, opera en modo no-op (sin errores).
# Referencia: https://langfuse.com/docs/observability/sdk/overview#client-setup
from langfuse import get_client

langfuse = get_client()
langfuse.auth_check()

print("✅ Conectado a Langfuse")
print(f"   SDK version: {langfuse._langfuse_sdk_version if hasattr(langfuse, '_langfuse_sdk_version') else 'v4'}")

✅ Conectado a Langfuse
   SDK version: v4


## 2. Anatomía de un buen System Prompt

Antes de crear versiones, entendamos qué hace un system prompt efectivo.
Un system prompt de producción para un agente tiene **5 componentes obligatorios**:

| Componente | Propósito | Ejemplo |
|-----------|-----------|---------|
| **1. Persona / Rol** | Define la identidad, tono y personalidad del agente | `Eres Alex, asistente de TechShop. Tono profesional y conciso.` |
| **2. Alcance (Scope)** | Define qué puede y qué NO puede hacer. Lo que no está en la lista, está implícitamente prohibido | `SOLO ayuda con productos y políticas de TechShop` |
| **3. Instrucciones de herramientas** | Cómo y cuándo usar las tools disponibles | `SIEMPRE usa search_catalog antes de responder sobre productos` |
| **4. Formato de respuesta** | Estructura, longitud, idioma esperado del output | `Máximo 3 frases. Incluye precio. Viñetas para listas.` |
| **5. Reglas anti-alucinación** | Qué hacer cuando no hay datos — con respuestas literales | `Si search_catalog devuelve vacío, di EXACTAMENTE: "No encontré productos..."` |

> **Principio de diseño: "Explicit beats Implicit".** Los LLMs son excelentes en seguir instrucciones, pero "rellenan" lo que no está especificado. Cada gap en el prompt es una oportunidad para la variabilidad no deseada. Ser explícito en las 5 dimensiones reduce esa variabilidad.

### El prompt actual (hardcodeado en `config.py`) — análisis de gaps

```python
SYSTEM_PROMPT = """Eres un asistente de atención al cliente para TechShop...
    SIEMPRE usa las herramientas disponibles para buscar información...
    Si no encuentras la información, dilo honestamente."""
```

**Gaps identificados:**

1. ❌ **Sin persona nombrada** — "un asistente" es anónimo; una persona con nombre genera más confianza y consistencia
2. ❌ **Sin límite de ámbito explícito** — no dice qué hacer si alguien pregunta por la capital de Francia
3. ❌ **Sin instrucción de fallback clara** — "dilo honestamente" es ambiguo; ¿qué dice exactamente?
4. ❌ **Sin formato de respuesta** — el agente puede responder de forma inconsistente entre llamadas
5. ❌ **Sin restricción anti-alucinación** — no prohíbe explícitamente inventar datos

Este análisis nos guía para construir v1 → v2 → v3.

In [3]:
# Prompt v1: el prompt hardcodeado actual
# Tomado directamente de src/techshop_agent/config.py
# Es funcional pero tiene los gaps identificados arriba.

PROMPT_V1 = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

Tu trabajo es ayudar a los clientes con:
- Consultas sobre productos (precios, disponibilidad, especificaciones)
- Preguntas frecuentes (envíos, devoluciones, garantías, pagos, horarios)

SIEMPRE usa las herramientas disponibles para buscar información antes de responder.
Responde en español, de forma concisa y profesional.
Si no encuentras la información, dilo honestamente.
"""

# Nombre del prompt en Langfuse — constante compartida
PROMPT_NAME = "techshop-system-prompt"

print("Prompt v1 (base):")
print("-" * 60)
print(PROMPT_V1)
print("-" * 60)
print("Gaps: sin persona, sin ámbito explícito, sin fallback específico, sin formato")


Prompt v1 (base):
------------------------------------------------------------
Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

Tu trabajo es ayudar a los clientes con:
- Consultas sobre productos (precios, disponibilidad, especificaciones)
- Preguntas frecuentes (envíos, devoluciones, garantías, pagos, horarios)

SIEMPRE usa las herramientas disponibles para buscar información antes de responder.
Responde en español, de forma concisa y profesional.
Si no encuentras la información, dilo honestamente.

------------------------------------------------------------
Gaps: sin persona, sin ámbito explícito, sin fallback específico, sin formato


## 3. Subir prompt v1 a Langfuse

Subimos el prompt actual como **primera versión** en Langfuse.
El contenido no cambia todavía — solo lo sacamos del código y lo ponemos en un sistema de gestión.

Esto ya resuelve el problema principal: **el prompt deja de vivir en el código**.

### Los tres conceptos fundamentales de Langfuse Prompt Management

**1. Prompt Name (nombre):** Identificador único del prompt. En nuestro caso: `"techshop-system-prompt"`.

**2. Version (versión):** Número secuencial automático e **inmutable**. Cada `create_prompt()` crea una nueva versión (v1, v2, v3...). **Una vez creada, una versión no se puede modificar ni eliminar** — esto garantiza reproducibilidad total.

**3. Label (etiqueta):** Puntero **movible** que señala a una versión específica:

| Label | Significado típico | Quién lo usa |
|-------|-------------------|-------------|
| `"production"` | La versión que leen los usuarios reales | El agente en producción |
| `"staging"` / `"latest"` | Candidata a reemplazar production | El equipo de evaluación |

**La idea clave:** El agente siempre pide el prompt por **label** (ej. `"production"`), nunca por número de versión. Cuando quieres cambiar el prompt, mueves el label a otra versión. El código no cambia.

Referencia: https://langfuse.com/docs/prompt-management/features/prompt-version-control

In [4]:
# Crear prompt v1 en Langfuse
# create_prompt() siempre crea una NUEVA versión inmutable.
# Si el nombre ya existe, no sobreescribe la versión anterior — añade una nueva.
# Referencia: https://langfuse.com/docs/prompt-management/features/prompt-version-control

try:
    langfuse.create_prompt(
        name=PROMPT_NAME,
        prompt=PROMPT_V1,
        labels=["latest", "production"],  # v1 es la versión activa inicial
        type="text",
        # config: metadata de modelo para referencia (no afecta al agente, solo legibilidad)
        config={"iteration": "v1", "changes": "baseline"},
    )
    print(f"✅ Prompt '{PROMPT_NAME}' v1 creado en Langfuse")
    print("   Labels: latest, production")
except Exception as e:
    # Puede ocurrir si el prompt ya existe con el mismo contenido exacto
    print(f"ℹ️  {e}")
    print("   El prompt ya existe — continuamos con el existente")

# Verificar que podemos leerlo de vuelta
check = langfuse.get_prompt(PROMPT_NAME, label="production", cache_ttl_seconds=0)
print(f"\n✅ Verificación: versión={check.version}, is_fallback={check.is_fallback}")


✅ Prompt 'techshop-system-prompt' v1 creado en Langfuse
   Labels: latest, production

✅ Verificación: versión=4, is_fallback=False


## 4. Usar el prompt desde Langfuse en el agente

En vez de hardcodear el prompt, lo **obtenemos de Langfuse** por label.

### ¿Por qué pedir por label y no por versión?

```python
# ❌ Esto acopla tu código a una versión específica:
prompt = langfuse.get_prompt("techshop-system-prompt", version=3)
# → Para cambiar de v3 a v4, necesitas modificar código y re-deployar.

# ✅ Esto desacopla tu código de la versión:
prompt = langfuse.get_prompt("techshop-system-prompt", label="production")
# → Para cambiar de v3 a v4, mueves el label en Langfuse.
# → El código no cambia. No hay re-deploy.
```

> El label es la indirección que permite cambiar el prompt sin tocar código. Es el mismo concepto que un puntero — el código apunta al label, y el label apunta a la versión.

### Patrón de disponibilidad: `fallback` + `cache` + `.compile()`

| Patrón | Qué hace | Por qué |
|--------|----------|---------|
| `fallback=PROMPT_V1` | Si Langfuse no responde, usa un prompt básico local | Garantiza disponibilidad |
| `cache_ttl_seconds=0` | En notebooks: siempre lee el prompt actual (sin cache) | Para ver cambios inmediatos |
| `cache_ttl_seconds=60` | En producción: cachea 60s para no saturar Langfuse | Reduce latencia y requests |
| `.compile()` | Renderiza variables `{{...}}` y devuelve el texto final | Compatible con variables actuales y futuras |

Referencia: https://langfuse.com/docs/prompt-management/features/guaranteed-availability

In [5]:
from techshop_agent import create_agent


def create_agent_with_prompt(prompt_label: str = "production") -> tuple:
    """Crea un agente usando el prompt almacenado en Langfuse.

    Returns:
        Tuple (agent, prompt_client) — el cliente se guarda para linking con trazas.
    """
    # Patrón correcto v4:
    # 1. fallback=PROMPT_V1: si Langfuse no responde, el agente sigue funcionando
    # 2. cache_ttl_seconds=0: en notebooks queremos siempre el prompt más reciente
    #    En producción usa cache_ttl_seconds=60 para reducir requests a Langfuse
    # 3. .compile(): renderiza {{variables}} si las hay, o devuelve texto plano
    #
    # Referencia: https://langfuse.com/docs/prompt-management/features/caching
    prompt_client = langfuse.get_prompt(
        PROMPT_NAME,
        label=prompt_label,
        fallback=PROMPT_V1,          # garantía de disponibilidad
        cache_ttl_seconds=0,         # siempre fresco en notebooks
    )

    system_prompt = prompt_client.compile()  # ← .compile() NO .prompt

    print(f"Prompt: '{PROMPT_NAME}'")
    print(f"  label={prompt_label!r}, version={prompt_client.version}")
    print(f"  is_fallback={prompt_client.is_fallback}")
    print(f"  texto-preview: {system_prompt[:80].strip()!r}...")

    return create_agent(system_prompt=system_prompt), prompt_client


# Crear agente con prompt v1 (label=production)
agent_v1, prompt_client_v1 = create_agent_with_prompt("production")
print("\n✅ Agente creado con prompt de Langfuse")


Prompt: 'techshop-system-prompt'
  label='production', version=4
  is_fallback=False
  texto-preview: 'Eres un asistente de atención al cliente para TechShop, una tienda online de ele'...

✅ Agente creado con prompt de Langfuse


In [6]:
# Probar con v1 — documentar comportamiento base
# Estas queries están diseñadas para exponer los gaps del prompt v1:
# - Pregunta de fuera del ámbito: ¿responde correctamente?  (gap: sin límite de ámbito)
# - Pregunta ambigua: ¿usa las herramientas?               (gap: sin instrucción clara)
# - Pregunta FAQ: ¿es la respuesta exacta y profesional?   (gap: sin formato)

test_queries = [
    "¿Qué portátiles tenéis?",
    "¿Puedo pagar en cuotas?",
    "¿Cuál es la capital de Francia?",         # out-of-scope: ¿lo rechaza?
    "Necesito algo para hacer fotos profesionales",
]

print("═" * 70)
print("Resultados con prompt v1 (baseline):")
print("═" * 70)

for q in test_queries:
    print(f"\nQ: {q}")
    r = str(agent_v1(q))
    preview = r[:200] + ("…" if len(r) > 200 else "")
    print(f"A: {preview}")

langfuse.flush()
print("\n✅ Trazas enviadas a Langfuse")


══════════════════════════════════════════════════════════════════════
Resultados con prompt v1 (baseline):
══════════════════════════════════════════════════════════════════════

Q: ¿Qué portátiles tenéis?
Voy a buscar los portátiles disponibles en nuestro catálogo.
Tool #1: search_catalog
Tenemos los siguientes portátiles disponibles:

1. **ProBook X1** - €1.199,99
   - Portátil profesional de 14 pulgadas
   - 16GB RAM y 512GB SSD
   - Stock: 12 unidades

2. **NanoBook Air 13** - €999,50
   - Portátil ultraligero para trabajo diario
   - Batería de larga duración
   - Stock: 8 unidades

¿Te interesa conocer más detalles sobre alguno de estos modelos o necesitas ayuda con otra cosa?A: Tenemos los siguientes portátiles disponibles:

1. **ProBook X1** - €1.199,99
   - Portátil profesional de 14 pulgadas
   - 16GB RAM y 512GB SSD
   - Stock: 12 unidades

2. **NanoBook Air 13** - €999,…

Q: ¿Puedo pagar en cuotas?
Voy a consultarte las opciones de pago disponibles.
Tool #2: get_faq_answer

## 5. Prompt v2 — Mejoras con las 5 secciones

### ¿Qué falló con v1 en las pruebas anteriores?

Con v1, el agente probablemente:
- Respondió la pregunta de fuera del ámbito ("capital de Francia") en lugar de rechazarla
- Usó un tono o formato inconsistente entre respuestas
- No indicó de forma determinista qué hacer cuando las herramientas no encuentran nada

### Cambios en v2 — cada sección aborda un gap

| Sección del prompt | Cambio | Gap de v1 que resuelve | Impacto esperado |
|--------|--------|-----------|-----------------|
| **1. Persona** | Nombre "Alex" | v1 era "un asistente" anónimo | Tono más consistente |
| **2. Alcance** | Lista explícita de lo que puede hacer | v1 no decía qué rechazar | Menos respuestas out-of-scope |
| **3. Herramientas** | "SIEMPRE usa tools ANTES de responder" | v1 era ambiguo | Modelo obligado a consultar datos |
| **4. Formato** | Viñetas + precio + disponibilidad | v1 no especificaba formato | Respuestas más útiles |
| **5. Anti-alucinación** | `NUNCA inventes precios, stock ni políticas` + respuesta literal de fallback | v1 decía "dilo honestamente" (ambiguo) | Menor riesgo de invención |

### Principio de diseño: "Explicit beats Implicit"

Los LLMs son excelentes en seguir instrucciones, pero "rellenan" lo que no está especificado.
Cada gap en el prompt es una oportunidad para la variabilidad no deseada.
Ser explícito en las 5 dimensiones (Persona, Alcance, Herramientas, Formato, Anti-alucinación) reduce esa variabilidad.

In [7]:
# Prompt v2: con persona, ámbito explícito, fallback literal y anti-alucinación
# Compara línea a línea con v1 para ver qué cambió y por qué.

PROMPT_V2 = """Eres Alex, el asistente de atención al cliente de TechShop, una tienda online de electrónica.

## Ámbito
SOLO puedes ayudar con:
- Consultas sobre productos del catálogo de TechShop (precios, disponibilidad, especificaciones)
- Preguntas frecuentes de TechShop (envíos, devoluciones, garantías, pagos, horarios de atención)

## Reglas de operación
1. SIEMPRE usa las herramientas disponibles (`search_catalog`, `get_faq_answer`) ANTES de responder.
2. SOLO responde con información que las herramientas devuelvan — NUNCA inventes precios, stock ni políticas.
3. Si las herramientas no devuelven resultados relevantes, di exactamente:
   "No he encontrado esa información en nuestro sistema. Te recomiendo contactar con soporte."
4. Si la consulta NO es sobre productos o políticas de TechShop, di exactamente:
   "Lo siento, solo puedo ayudarte con consultas sobre productos y políticas de TechShop."

## Formato de respuesta
- Sé conciso y profesional
- Para listas de productos: usa viñetas con nombre, precio y disponibilidad
- Responde siempre en español
"""

print("Prompt v2 (mejorado):")
print("-" * 60)
print(PROMPT_V2)
print("-" * 60)
print("\nMejoras v1→v2:")
print("  ✅ Persona nombrada (Alex)")
print("  ✅ Ámbito de actuación explícito")
print("  ✅ Respuesta de fallback literal (no ambigua)")
print("  ✅ Respuesta out-of-scope literal")
print("  ✅ Restricción anti-alucinación explícita")
print("  ✅ Formato de respuesta definido")


Prompt v2 (mejorado):
------------------------------------------------------------
Eres Alex, el asistente de atención al cliente de TechShop, una tienda online de electrónica.

## Ámbito
SOLO puedes ayudar con:
- Consultas sobre productos del catálogo de TechShop (precios, disponibilidad, especificaciones)
- Preguntas frecuentes de TechShop (envíos, devoluciones, garantías, pagos, horarios de atención)

## Reglas de operación
1. SIEMPRE usa las herramientas disponibles (`search_catalog`, `get_faq_answer`) ANTES de responder.
2. SOLO responde con información que las herramientas devuelvan — NUNCA inventes precios, stock ni políticas.
3. Si las herramientas no devuelven resultados relevantes, di exactamente:
   "No he encontrado esa información en nuestro sistema. Te recomiendo contactar con soporte."
4. Si la consulta NO es sobre productos o políticas de TechShop, di exactamente:
   "Lo siento, solo puedo ayudarte con consultas sobre productos y políticas de TechShop."

## Formato de r

In [8]:
# Subir prompt v2 a Langfuse
# Label "latest" únicamente — "production" sigue apuntando a v1.
# Esto es el flujo correcto: candidato a producción, pendiente de evaluación.
# Referencia: https://langfuse.com/docs/prompt-management/features/prompt-version-control

try:
    langfuse.create_prompt(
        name=PROMPT_NAME,
        prompt=PROMPT_V2,
        labels=["latest"],  # solo latest; production queda en v1 hasta que evaluemos
        type="text",
        config={"iteration": "v2", "changes": "persona, scope, fallback, anti-hallucination, format"},
    )
    print(f"✅ Prompt '{PROMPT_NAME}' v2 creado en Langfuse")
    print("   Labels: latest")
    print("   production: sigue en v1 ← no cambiamos producción todavía")
except Exception as e:
    print(f"ℹ️  {e}")


✅ Prompt 'techshop-system-prompt' v2 creado en Langfuse
   Labels: latest
   production: sigue en v1 ← no cambiamos producción todavía


## 6. Comparar v1 (production) vs v2 (latest)

Usamos el mismo conjunto de queries para las dos versiones.
Las queries están diseñadas para probar los gaps que identificamos en v1:

| Query | Gap que prueba |
|-------|---------------|
| "¿Cuál es la capital de Francia?" | ¿Rechaza correctamente lo out-of-scope? |
| "Necesito algo para escuchar música" | ¿Usa herramientas? ¿Es el formato mejor? |
| "¿Tenéis iPhone?" | ¿Responde el no-encontrado con la frase literal correcta? |
| "¿Cuál es la política de devoluciones?" | ¿Usa `get_faq_answer` correctamente? |

### Hipótesis de mejora en v2
- **Francia** → v1 responderá; v2 dará la frase de rechazo literal
- **Música** → v2 listará productos con precio y disponibilidad (formato definido)
- **iPhone** → v2 dará el fallback literal, no una respuesta inventada
- **Devoluciones** → ambas deberían responder, pero v2 tendrá formato más consistente


In [9]:
# Crear agente con v2
agent_v2, prompt_client_v2 = create_agent_with_prompt("latest")

comparison_queries = [
    "¿Cuál es la capital de Francia?",
    "Necesito algo para escuchar música",
    "¿Tenéis iPhone?",
    "¿Cuál es la política de devoluciones?",
]

print("\nComparación v1 (production) vs v2 (latest):")

for q in comparison_queries:
    print(f"\n{'═' * 70}")
    print(f"Query: {q}")
    print("═" * 70)

    r1 = str(agent_v1(q))
    r2 = str(agent_v2(q))

    max_len = 250
    print(f"\n  v1: {r1[:max_len]}{'…' if len(r1) > max_len else ''}")
    print(f"\n  v2: {r2[:max_len]}{'…' if len(r2) > max_len else ''}")

langfuse.flush()
print("\n✅ Trazas enviadas — compara ambas en Langfuse UI bajo la misma sesión")


Prompt: 'techshop-system-prompt'
  label='latest', version=5
  is_fallback=False
  texto-preview: 'Eres Alex, el asistente de atención al cliente de TechShop, una tienda online de'...

Comparación v1 (production) vs v2 (latest):

══════════════════════════════════════════════════════════════════════
Query: ¿Cuál es la capital de Francia?
══════════════════════════════════════════════════════════════════════
Como te mencioné antes, no puedo ayudarte con preguntas generales. Soy un asistente especializado de TechShop dedicado a ayudarte con productos y servicios de electrónica.

¿Hay algo más que necesites sobre la **PixelCam Z** o cualquier otro producto de TechShop?Lo siento, solo puedo ayudarte con consultas sobre productos y políticas de TechShop.

Si tienes dudas sobre nuestros productos electrónicos, políticas de envío, devoluciones, garantías o pagos, estaré encantado de ayudarte. 😊
  v1: Como te mencioné antes, no puedo ayudarte con preguntas generales. Soy un asistente especiali

## 7. Promover v2 a producción

Si las pruebas son satisfactorias, promovemos v2 moviendo el label `production`.

### El flujo de deploy de un prompt (análogo a Blue/Green deployment)

```
Estado inicial:
  v1 [production, latest] ← activa, los usuarios la usan

Paso 1: Crear v2 con mejoras
  langfuse.create_prompt(name, prompt=PROMPT_V2, labels=["latest"])
  Estado: v1 [production] | v2 [latest]

Paso 2: Evaluar v2 contra v1
  - Ejecutar el dataset de test con v2
  - Comparar scores: accuracy, relevance, hallucination_rate
  - Si v2 ≥ v1 → continuar. Si v2 < v1 → descartar.

Paso 3: Promover v2 a producción
  - Mover label "production" de v1 a v2
  Estado: v1 [] | v2 [production, latest]
  - El agente lee v2 automáticamente en la siguiente request. Sin deploy.

Paso 4 (si hay problemas): Rollback
  - Mover label "production" de vuelta a v1
  Estado: v1 [production] | v2 [latest]
  - Tiempo de rollback: segundos. Sin deploy. Sin commit.
```

> Este flujo es análogo a Blue/Green deployment pero para prompts. En todo momento hay una versión activa, y el cambio es instantáneo y reversible.

### Compare con el enfoque hardcodeado

| Acción | Con labels (Langfuse) | Sin labels (hardcoded) |
|--------|----------------------|----------------------|
| **Deploy** | Mover label (segundos) | Commit → PR → CI → deploy (minutos/horas) |
| **Rollback** | Mover label a versión anterior (segundos) | Revert commit → re-deploy (minutos/horas) |
| **Evaluación A/B** | Dos labels: "production" + "staging" | Requiere feature flags |

Referencia: https://langfuse.com/docs/prompt-management/features/prompt-version-control

In [10]:
# Promover v2 a production
# Creamos una nueva versión con el mismo contenido pero con ambos labels.
# En un pipeline real, este paso estaría gatekeado por la evaluación automática.

try:
    langfuse.create_prompt(
        name=PROMPT_NAME,
        prompt=PROMPT_V2,
        labels=["latest", "production"],   # ← v2 ahora es production
        type="text",
        config={"iteration": "v2", "status": "promoted"},
    )
    print("✅ Prompt v2 promovido a production")
except Exception as e:
    print(f"ℹ️  {e}")

# Verificar el estado actual — label production debe apuntar al texto de v2
current = langfuse.get_prompt(PROMPT_NAME, label="production", cache_ttl_seconds=0)
print(f"\nEstado actual:")
print(f"  production → versión {current.version}")
print(f"  preview: {current.compile()[:80].strip()!r}…")

✅ Prompt v2 promovido a production

Estado actual:
  production → versión 6
  preview: 'Eres Alex, el asistente de atención al cliente de TechShop, una tienda online de'…


## 8. Rollback — Revertir en segundos

Si v2 causa problemas en producción, el rollback es inmediato:

### Ejemplo de rollback en producción

```
Lunes 09:00  — v1 [production]. Todo funciona. Score promedio: 0.88
Lunes 14:00  — Se crea v2 con mejoras. Se prueban con 50 queries. Score: 0.91
Lunes 16:00  — Se promueve v2 a producción.
Martes 10:00 — Los usuarios reportan respuestas demasiado largas.
Martes 10:05 — Rollback: se crea nueva versión con contenido de v1 + label "production".
              En la siguiente request, el agente ya usa v1. Sin deploy.
```

```python
# Para rollback: crear nueva versión con contenido de v1 + label production
langfuse.create_prompt(
    name=PROMPT_NAME,
    prompt=PROMPT_V1,        # contenido de la versión anterior
    labels=["production"],   # vuelve a ser el prompt activo
    type="text",
    config={"status": "rollback-from-v2"},
)
```

> **¿Por qué crear una nueva versión en lugar de editar la existente?**
> Los prompts en Langfuse son **inmutables por diseño**. Cada versión es un artefacto histórico que puedes auditar, comparar y referenciar desde las trazas. Modificar una versión existente rompería la trazabilidad. El rollback es simplemente crear una nueva versión con el contenido antiguo y moverle el label `production`.

Referencia: https://langfuse.com/docs/prompt-management/features/prompt-version-control

## 9. Variables en Prompts — Parametrizar en Tiempo de Ejecución

Las variables permiten **personalizar** el prompt en cada llamada sin crear nuevas versiones.
Usa la sintaxis `{{nombre_variable}}` en Langfuse y llama `.compile(nombre_variable=valor)` en Python.

### Cuándo usar variables

| Caso de uso | Variable | Ejemplo |
|-------------|---------|---------|
| **Multi-tienda** | `{{store_name}}` | Adaptar el agente a distintas tiendas: "TechShop", "AudioWorld" |
| **Multi-idioma** | `{{language}}` | "español", "English", "português" |
| **Configuración** | `{{max_results}}` | "3", "5", "10" |
| **Feature flags** | `{{include_prices}}` | "Incluye precios" / "" |

Las variables separan el contenido estructural del prompt (que se versiona) de los valores contextuales (que se inyectan en runtime). Esto evita crear versiones duplicadas para cada combinación de tienda/idioma.

Referencia: https://langfuse.com/docs/prompt-management/features/variables

In [14]:
# Prompt v3: con variable {{store_name}} para demostrar parametrización
# La variable se sustituye en tiempo de ejecución con .compile()

PROMPT_V3_TEMPLATE = """Eres Alex, el asistente de atención al cliente de {{store_name}}, una tienda online de electrónica.

## Ámbito
SOLO puedes ayudar con:
- Consultas sobre productos del catálogo de {{store_name}} (precios, disponibilidad, especificaciones)
- Preguntas frecuentes de {{store_name}} (envíos, devoluciones, garantías, pagos, horarios)

## Reglas de operación
1. SIEMPRE usa las herramientas disponibles (`search_catalog`, `get_faq_answer`) ANTES de responder.
2. SOLO responde con información que las herramientas devuelvan — NUNCA inventes precios, stock ni políticas.
3. Si las herramientas no devuelven resultados, di: "No he encontrado esa información en {{store_name}}."
4. Si la consulta no es sobre {{store_name}}, di: "Solo puedo ayudarte con consultas sobre {{store_name}}."

## Formato
- Conciso y profesional. Viñetas para listas de productos con precio y disponibilidad.
- Responde siempre en español.
"""

# Subir v3 a Langfuse
try:
    langfuse.create_prompt(
        name=PROMPT_NAME,
        prompt=PROMPT_V3_TEMPLATE,
        labels=["latest"],
        type="text",
        config={"iteration": "v3", "changes": "added store_name variable"},
    )
    print("✅ Prompt v3 con variable {{store_name}} creado")
except Exception as e:
    print(f"ℹ️  {e}")

# Compilar con variable — .compile() sustituye {{store_name}} por el valor

store_name = "Paquita's Churrería Supreme"
prompt_v3_client = langfuse.get_prompt(PROMPT_NAME, label="latest", cache_ttl_seconds=0)
compiled_prompt = prompt_v3_client.compile(store_name=store_name)

print(f"\nPlantilla (raw): {PROMPT_V3_TEMPLATE[:80].strip()!r}…")
print(f"Compilado:       {compiled_prompt[:80].strip()!r}…")

# Verificar que la variable fue sustituida
assert "{{store_name}}" not in compiled_prompt, "La variable no fue sustituida"
assert store_name in compiled_prompt
print(f"\n✅ Variable sustituida correctamente por .compile(store_name='{store_name}')")


✅ Prompt v3 con variable {{store_name}} creado

Plantilla (raw): 'Eres Alex, el asistente de atención al cliente de {{store_name}}, una tienda onl'…
Compilado:       "Eres Alex, el asistente de atención al cliente de Paquita's Churrería Supreme, u"…

✅ Variable sustituida correctamente por .compile(store_name='Paquita's Churrería Supreme')


## 10. Vincular Prompts a Trazas — Medir el impacto de cada versión

Hasta ahora, las trazas en Langfuse muestran **qué respuesta generó el agente**, pero no saben **qué versión del prompt** se usó para generarla.

Vincular el prompt a la traza activa el **análisis por versión de prompt** en Langfuse UI:
- Latencia media por versión
- Coste (tokens) por versión
- Puntuaciones de evaluación por versión

**Esto es la clave del prompt manager:** demuestra que v2 es mejor que v1 con datos cuantitativos, no con intuición.

### Cómo funciona

```python
# 1. Obtén el prompt_client (no solo el texto)
prompt_client = langfuse.get_prompt(name, label="production")

# 2. Dentro de @observe, vincúlalo a la generation activa:
langfuse.update_current_generation(prompt=prompt_client)
# ↑ Esto registra prompt_id y versión en la traza
```

**Lo que ves después en el dashboard:**
```
Dashboard > Prompts > techshop-system-prompt
├── v1 (1,200 trazas) → Score: 0.85, Latencia: 1.8s, Coste: $0.003
└── v2 (800 trazas)   → Score: 0.91, Latencia: 2.1s, Coste: $0.004
```

> Sin esta vinculación, tendrías trazas por un lado y prompts por otro, sin conexión. Con ella, cada traza sabe qué versión del prompt se usó.

Referencia: https://langfuse.com/docs/prompt-management/features/link-to-traces

In [15]:
from langfuse import observe, propagate_attributes

@observe(name="techshop_query_with_prompt_link")
def run_with_prompt_link(user_query: str, prompt_label: str = "production") -> str:
    """Ejecuta el agente vinculando el prompt a la traza de Langfuse.

    El prompt_client object (no solo el texto compilado) se pasa a
    update_current_generation() para crear el vínculo prompt→traza.
    """
    # Obtener prompt_client (necesitamos el objeto, no solo el texto)
    prompt_client = langfuse.get_prompt(
        PROMPT_NAME,
        label=prompt_label,
        fallback=PROMPT_V1,
        cache_ttl_seconds=0,
    )
    compiled = prompt_client.compile()

    # Vincular el prompt a la generación activa
    # Nota: este método debe llamarse dentro de un span @observe activo
    # Referencia: https://langfuse.com/docs/prompt-management/features/link-to-traces
    langfuse.update_current_generation(prompt=prompt_client)

    agent = create_agent(system_prompt=compiled)

    with propagate_attributes(
        metadata={
            "prompt_label": prompt_label,
            "prompt_version": str(prompt_client.version),
        }
    ):
        response = agent(user_query)

    return str(response)


# Ejecutar una query con prompt linking activo
test_query = "¿Qué auriculares tenéis por menos de 100 euros?"
result = run_with_prompt_link(test_query, prompt_label="production")
print(f"Q: {test_query}")
print(f"A: {result[:300]}")

langfuse.flush()
print("\n✅ Traza enviada con prompt vinculado — busca 'prompt_version' en la UI de Langfuse")



Tool #1: search_catalog
La búsqueda anterior devolvió un producto de redes. Déjame hacer una búsqueda más específica:
Tool #2: search_catalog
No he encontrado esa información en nuestro sistema. Te recomiendo contactar con soporte.

Parece que en este momento no tenemos auriculares disponibles en nuestro catálogo o no se muestran en la búsqueda. Te sugiero que:
- Contactes con nuestro equipo de soporte para confirmar disponibilidad
- Navegues por las categorías de productos en nuestra tienda online
- Consultes de nuevo más adelante, ya que el catálogo se actualiza regularmenteQ: ¿Qué auriculares tenéis por menos de 100 euros?
A: No he encontrado esa información en nuestro sistema. Te recomiendo contactar con soporte.

Parece que en este momento no tenemos auriculares disponibles en nuestro catálogo o no se muestran en la búsqueda. Te sugiero que:
- Contactes con nuestro equipo de soporte para confirmar disponibilidad
- Nave

✅ Traza enviada con prompt vinculado — busca 'prompt_version

## 11. Módulo de Solución: `techshop_agent.solution.prompt_provider`

Todo lo que hemos construido en este notebook está encapsulado en el módulo de referencia:

```
src/techshop_agent/solution/prompt_provider.py
```

Este módulo implementa el ciclo completo de gestión de prompts como código de producción:

| Función | Descripción |
|---------|-------------|
| `get_system_prompt(label, cache_ttl_seconds)` | Fetch por label con fallback y caché |
| `get_prompt_client(label, cache_ttl_seconds)` | Fetch del client object para trace linking |
| `create_prompt_version(content, labels, config)` | Crear nueva versión con metadatos |
| `process_query_with_prompt(query, prompt_label, ...)` | Query completa con observabilidad + prompt linking |
| `FALLBACK_PROMPT` | Constante — fallback hardcodeado de emergencia |
| `PROMPT_NAME` | Constante — nombre del prompt en Langfuse |

### Cómo usar el módulo en el agente

```python
from techshop_agent.solution.prompt_provider import (
    get_system_prompt,
    process_query_with_prompt,
    FALLBACK_PROMPT,
)

# Fetch sin preocuparte por errores de red:
system_prompt = get_system_prompt(label="production")

# O la versión completa con observabilidad + prompt linking:
response = process_query_with_prompt(
    "¿Qué portátiles tenéis?",
    prompt_label="production",
    user_id="user-123",
    session_id="session-abc",
)
```

> Este módulo es el **estado objetivo** que los estudiantes deben alcanzar.
> El código del notebook es el camino; el módulo es el destino.


## 12. Vista previa: Evaluación de Prompts en CI/CD

### El siguiente paso

Ahora que podemos versionar prompts, la pregunta natural es: **¿cómo demostrar que v2 es mejor que v1 de forma reproducible y automatizable?**

Eso es exactamente lo que haremos en el Día 2:

```bash
Desarrollador crea prompt v3
        │
        ▼
create_prompt(v3, labels=["staging"])
        │
        ▼
CI/CD ejecuta evaluación automática:
  - Determinista: contains("TechShop"), not-contains("lo siento, no puedo")
  - LLM-as-judge: relevancia, fidelidad, tono profesional
        │
        ├── Todas las evaluaciones pasan ✅ → move label "production" a v3
        └── Alguna falla ❌ → bloquear, notificar, "production" sigue en v2
```

### Herramientas que veremos

- **Langfuse Datasets:** test cases almacenados en Langfuse, ejecutables desde Python
- **Evaluación determinista:** `contains`, `regex`, `not-contains` — rápidas y baratas
- **LLM-as-judge:** otro LLM evalúa relevancia, fidelidad y profesionalidad
- **Integration tests:** pytest con trazas reales verificadas vía `langfuse.api`

> En el **Notebook 4** implementaremos este pipeline completo — del prompt versión candidata a la decisión de deploy.


## Resumen

### Conceptos aprendidos

| Concepto | Qué aprendimos |
|----------|----------------|
| **Prompt drift** | Sin control, los prompts cambian sin registro → comportamiento impredecible |
| **Versión inmutable** | Cada `create_prompt()` crea artefacto histórico — nunca edita el anterior |
| **Labels** | `production`, `staging`, `latest` — punteros movibles a versiones concretas |
| **Fallback** | `get_prompt(fallback=...)` garantiza disponibilidad offline |
| **Caché** | `cache_ttl_seconds=60` en prod, `=0` en notebooks |
| **`.compile()`** | Siempre en lugar de `.prompt` — renderiza variables `{{...}}` |
| **Variables** | `{{variable}}` + `.compile(var=valor)` — personalización sin nuevas versiones |
| **Trace linking** | `update_current_generation(prompt=client)` → métricas por versión en Langfuse |
| **Rollback** | Crear nueva versión con contenido anterior + label `production` (segundos) |

### Anti-patrones a evitar

| Anti-patrón | Solución |
|-------------|----------|
| "Editar y rezar" — cambiar prompt sin evaluar | SIEMPRE evaluar en staging antes de promover |
| Instrucciones contradictorias ("Sé conciso" + "Sé detallado") | Revisión de coherencia antes de deploy |
| No tener fallback | SIEMPRE implementar `fallback=FALLBACK_PROMPT` |
| Pedir por versión en vez de label | SIEMPRE: `get_prompt(label="production")` |
| No vincular prompt a traza | SIEMPRE: `update_current_generation(prompt=prompt_client)` |

### API v4 (resumen)

```python
# Inicialización
from langfuse import get_client
langfuse = get_client()

# Leer prompt por label (SIEMPRE por label, nunca por versión)
prompt_client = langfuse.get_prompt(name, label="production", fallback=FALLBACK, cache_ttl_seconds=60)
system_prompt = prompt_client.compile()                     # sin variables
system_prompt = prompt_client.compile(store_name="TechShop") # con variables

# Crear nueva versión inmutable
langfuse.create_prompt(name, prompt=content, labels=["latest"], type="text")

# Vincular prompt a traza
langfuse.update_current_generation(prompt=prompt_client)
```

### Los 5 componentes del system prompt (checklist)

- [ ] **1. Persona / Rol** — nombre y identidad del agente
- [ ] **2. Alcance (Scope)** — qué puede y qué NO puede hacer (lista explícita)
- [ ] **3. Instrucciones de herramientas** — cuándo y cómo usar las tools
- [ ] **4. Formato de respuesta** — estructura, longitud, idioma
- [ ] **5. Reglas anti-alucinación** — `NUNCA inventes datos` + respuestas literales de fallback

### Prompt hardcodeado vs. Prompt Manager

| Aspecto | Hardcoded en código | Prompt Manager (Langfuse) |
|---------|--------------------|-----------------------------|
| **Dónde vive** | String en `config.py` | Servicio externo con API |
| **Deploy** | Commit → PR → CI → deploy | Mover label (segundos) |
| **Rollback** | Revert commit → re-deploy | Mover label (segundos) |
| **Métricas por versión** | No posible | Dashboard con scores por versión |
| **¿Cuándo está bien?** | Prototipos y notebooks | Producción con equipo |

> **Del "edito el prompt y rezo" al "versiono, evalúo, y despliego con confianza."**

---

## Siguiente: Notebook 4 — Evaluación Automática del Agente